## 🗣️ General Prompting on Instruct model

In [ ]:
import os
import time

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from datasets import load_dataset

### 📦 Dataset Prep

In [17]:
ds = load_dataset("AI4Sec/cti-bench", "cti-mcq")
data = ds['test'] if 'test' in ds else ds['train']

# Basic information
print(f"Number of examples: {len(data)}")
print(f"Column names: {data.column_names}")
print(f"Features: {data.features}")
print()

# Show first example
print("First example:")
print(data[0])

Number of examples: 2500
Column names: ['URL', 'Question', 'Option A', 'Option B', 'Option C', 'Option D', 'Prompt', 'GT']
Features: {'URL': Value('string'), 'Question': Value('string'), 'Option A': Value('string'), 'Option B': Value('string'), 'Option C': Value('string'), 'Option D': Value('string'), 'Prompt': Value('string'), 'GT': Value('string')}

First example:
{'URL': 'https://attack.mitre.org/techniques/T1548/', 'Question': "Which of the following mitigations involves preventing applications from running that haven't been downloaded from legitimate repositories?", 'Option A': 'Audit', 'Option B': 'Execution Prevention', 'Option C': 'Operating System Configuration', 'Option D': 'User Account Control', 'Prompt': "You are given a multiple-choice question (MCQ) from a Cyber Threat Intelligence (CTI) knowledge benchmark dataset. Your task is to choose the best option among the four provided. Return your answer as a single uppercase letter: A, B, C, or D.  **Question:** Which of the f

In [18]:
data_subset = data.select(range(100))

### ⚙️ MODEL CONFIGURATION - CHANGE HERE TO SWITCH MODELS

In [19]:
# ========================================
# 🔧 MAIN CONFIGURATION - LOCAL MODEL
# ========================================

# Model Configuration for LOCAL execution
# MODEL_PATH = 'Qwen/Qwen2.5-0.5B-Instruct'  # HuggingFace model path

# Alternative local models you can try:
MODEL_NAME = "Qwen"
MODEL_PATH = 'Qwen/Qwen2.5-1.5B-Instruct' 

# Generation Parameters
TEMPERATURE = 0.01  # Transformers doesn't support 0, use very low value
TOP_P = 1
SEED = 42
MAX_TOKENS = 2048

# System Prompt
SYSTEM_PROMPT = 'You are a cybersecurity expert specializing in cyberthreat intelligence.'

# Device Configuration
USE_GPU = True  # Set to False if you don't have GPU
LOAD_IN_8BIT = False  # Set to True to save memory (requires bitsandbytes)

print(f"✓ Configuration loaded")
print(f"  Path: {MODEL_PATH}")
print(f"  Use GPU: {USE_GPU}")
print(f"  8-bit loading: {LOAD_IN_8BIT}")

✓ Configuration loaded
  Path: Qwen/Qwen2.5-1.5B-Instruct
  Use GPU: True
  8-bit loading: False


In [20]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Determine device
if USE_GPU and torch.cuda.is_available():
    device = "cuda"
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("✓ Using CPU")

# Load model
if LOAD_IN_8BIT and device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        load_in_8bit=True,
        device_map="auto",
        dtype=torch.float16
    )
    print("✓ Model loaded in 8-bit mode")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        dtype=torch.float16 if device == "cuda" else torch.float32
    )
    model = model.to(device)
    print(f"✓ Model loaded on {device}")

print("✓ Model ready for inference!")

✓ Using GPU: NVIDIA GeForce RTX 4050 Laptop GPU


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


✓ Model loaded on cuda
✓ Model ready for inference!


### 🧠 Prediction Method

In [21]:
def get_single_prediction(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=TEMPERATURE > 0,
            pad_token_id=tokenizer.eos_token_id
        )

    # PERBAIKAN: Hanya decode bagian yang baru di-generate
    # Potong input_ids dari output_ids
    input_length = inputs['input_ids'].shape[1]
    generated_ids = outputs[0][input_length:]

    # Decode hanya bagian yang di-generate
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    print(f"Response: '{response}'")
    return response

def format_mcq(text):
    """Format Multiple Choice Question (extract A/B/C/D)"""
    last_line = text.split('\n')[-1].rstrip()

    if last_line.startswith('A)') or last_line.startswith('B)') or last_line.startswith('C)') or last_line.startswith('D)'):
        return last_line[0]
    if last_line.endswith('A') or last_line.endswith('B') or last_line.endswith('C') or last_line.endswith('D'):
        return last_line[-1]
    if last_line.endswith('**'):
        return last_line[-3]
    if len(last_line) == 0:
        last_line = text.split('\n')[-2].rstrip()
        if last_line.startswith('A)') or last_line.startswith('B)') or last_line.startswith('C)') or last_line.startswith('D)'):
            return last_line[0]
        if last_line.endswith('A') or last_line.endswith('B') or last_line.endswith('C') or last_line.endswith('D'):
            return last_line[-1]
        if last_line.endswith('**'):
            return last_line[-3]
    return ' '.join(text.split('\n'))

print("✓ Formatting functions loaded")

def run_evaluation(task, save_to_drive=True):
    """
    Run evaluation on a CTI-Bench task using HuggingFace datasets.

    Args:
        task: One of 'cti-mcq', 'cti-rcm', 'cti-vsp', 'cti-taa'
    """
    print(f"\n{'='*60}")
    print(f"Starting evaluation: {task.upper()}")
    print(f"Model: {MODEL_NAME}")
    print(f"Dataset: AI4Sec/cti-bench")
    print(f"{'='*60}\n")

    # Determine which column contains the prompts
    # Common column names: 'Prompt', 'prompt', 'question', 'input', 'text'
    prompt_column = None
    for col in ['Prompt', 'prompt', 'question', 'input', 'text']:
        if col in data_subset.column_names:
            prompt_column = col
            break

    if prompt_column is None:
        print(f"Available columns: {data_subset.column_names}")
        raise ValueError("Could not find prompt column. Please specify manually.")

    print(f"Using column '{prompt_column}' for prompts\n")

    # Track metrics
    start_time = time.time()
    count_chars = 0
    instructions_failed = 0

    # Storage for results
    all_responses = []
    all_results = []
    task_type = task.split('-')[-1]
    # Process each example
    for index, example in enumerate(data_subset):
        prompt = example[prompt_column]
        print(f"prompt : {prompt}")
        try:
            print("==============inside the try in run evaluation===============")
            output = get_single_prediction(prompt)
            count_chars += len(output)
            all_responses.append(output)
            print(f"output: {output}")
            print(f"count_chars: {count_chars}")
            # Format output based on task type
            if task_type == 'mcq':
                answer = format_mcq(output)
            else:
                raise ValueError(f'Unknown task type: {task_type}')

        except Exception as e:
            answer = 'Error'
            all_responses.append(answer)
            print(f'❌ Exception at example {index+1}: {e}')

        all_results.append(answer)
        print(f'{index+1}/{len(data_subset)}: {answer}')

    # Calculate metrics
    time_taken = time.time() - start_time

    print(f"\n{'='*60}")
    print("EVALUATION COMPLETE")
    print(f"{'='*60}")
    print(f"Time taken: {time_taken:.2f} seconds ({time_taken/60:.2f} minutes)")
    print(f"Characters generated: {count_chars:,}")
    print(f"Instructions failed: {instructions_failed}/{len(data_subset)}")
    print(f"Success rate: {(len(data_subset)-instructions_failed)/len(data_subset)*100:.1f}%")

    # Save results
    if save_to_drive:
        # Check if Drive is mounted
        if not os.path.exists('/content/drive/MyDrive'):
            print("\n⚠️  Google Drive not mounted. Mounting now...")
            drive.mount('/content/drive')

        # Create directory in Google Drive
        output_dir = '/content/drive/MyDrive/Colab Notebooks/CTI-Bench-Results'
        os.makedirs(output_dir, exist_ok=True)
        print(f"\n💾 Saving to Google Drive: {output_dir}")
    else:
        # Save to Colab temporary storage
        output_dir = 'results'
        os.makedirs(output_dir, exist_ok=True)
        print(f"\n💾 Saving to Colab storage: {output_dir}")

    suffix = f"_first{len(data_subset)}"
    output_dir = './results'
    out_response = f"{output_dir}/{task}_{MODEL_NAME}{suffix}_response.txt"
    out_result = f"{output_dir}/{task}_{MODEL_NAME}{suffix}_result.txt"

    with open(out_response, 'w', encoding='utf-8') as f:
        out_str = ''
        for i in range(len(all_responses)):
            out_str += f'#####{i+1}#####\n'
            out_str += all_responses[i]
            out_str += '\n\n'
        f.write(out_str)

    with open(out_result, 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_results))

    print(f"\n📁 Results saved:")
    print(f"  - Full responses: {out_response}")
    print(f"  - Formatted results: {out_result}")
    print(f"\n{'='*60}\n")

    return all_results

print("✓ Evaluation function ready")

✓ Formatting functions loaded
✓ Evaluation function ready


In [22]:
run_evaluation('cti-mcq', False)


Starting evaluation: CTI-MCQ
Model: Qwen
Dataset: AI4Sec/cti-bench

Using column 'Prompt' for prompts

prompt : You are given a multiple-choice question (MCQ) from a Cyber Threat Intelligence (CTI) knowledge benchmark dataset. Your task is to choose the best option among the four provided. Return your answer as a single uppercase letter: A, B, C, or D.  **Question:** Which of the following mitigations involves preventing applications from running that haven't been downloaded from legitimate repositories?  **Options:** A) Audit B) Execution Prevention C) Operating System Configuration D) User Account Control  **Important:** The last line of your answer should contain only the single letter corresponding to the best option, with no additional text. 
==============inside the try in run evaluation===============
Response: 'B'
output: B
count_chars: 1
1/100: B
prompt : You are given a multiple-choice question (MCQ) from a Cyber Threat Intelligence (CTI) knowledge benchmark dataset. Your ta

['B',
 'C',
 'C',
 'A',
 'D',
 'B',
 'D',
 'A',
 'D',
 'D',
 'D',
 'B',
 'A',
 'B',
 'C',
 'B',
 'A',
 'B',
 'C',
 'D',
 'C',
 'A',
 'B',
 'C',
 'A',
 'C',
 'C',
 'A',
 'B',
 'A',
 'A',
 'B',
 'A',
 'D',
 'C',
 'D',
 'C',
 'B',
 'B',
 'C',
 'C',
 'D',
 'A',
 'C',
 'C',
 'C',
 'D',
 'B',
 'A',
 'B',
 'C',
 'B',
 'C',
 'D',
 'C',
 'A',
 'A',
 'B',
 'C',
 'C',
 'A',
 'C',
 'B',
 'A',
 'B',
 'C',
 'C',
 'A',
 'B',
 'C',
 'A',
 'A',
 'C',
 'A',
 'A',
 'A',
 'B',
 'B',
 'C',
 'B',
 'B',
 'B',
 'A',
 'A',
 'C',
 'C',
 'C',
 'B',
 'D',
 'C',
 'C',
 'B',
 'C',
 'B',
 'D',
 'B',
 'A',
 'A',
 'A',
 'D']